In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

IMPORTING DATASETS

In [2]:
data_log2 = pd.read_pickle('data_log2_Lisbon_Coimbra_threshold.pkl')

In [3]:
data_log2

Protein.Group,A0A0A0MS15,A0A0B4J1X8,A0A0C4DH38,O00391,O00533,O14498,O14594,O14773,O15394,O43505,...,Q9NRN5,Q9NT99,Q9P121,Q9P2S2,Q9UBP4,Q9UBX5,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7
LIS090,7.524839,9.440618,7.131415,6.264067,8.355800,6.222868,5.798631,5.355221,7.860578,9.961581,...,4.881704,6.206844,8.309122,6.827273,10.918669,6.753711,7.058630,7.842011,5.686817,8.661849
LIS098,7.791541,10.339338,7.438942,5.908674,7.962676,5.939913,5.244541,5.406958,7.345982,10.132975,...,4.443076,5.911802,8.087309,6.454133,10.520913,6.709801,6.675844,7.766999,5.921993,7.172317
LIS017,6.309827,10.338246,6.398735,5.907285,8.112664,5.983963,6.077408,5.565893,7.673055,10.326800,...,5.340762,5.799201,8.334036,6.557727,10.662704,7.273665,6.746998,7.561639,6.478499,6.753203
LIS007,9.705954,11.126569,8.909980,6.499706,7.305670,6.535481,4.455360,5.277166,7.376681,9.405173,...,4.832850,5.664161,7.650348,6.108621,10.128703,6.935637,6.173705,6.919900,6.039838,8.260840
LIS026,8.114325,11.020813,8.179367,6.765097,8.106155,5.791983,5.600198,5.660937,7.862365,9.843741,...,5.365864,5.873333,8.227313,6.172213,10.573581,7.251549,6.740347,7.526218,5.885477,7.837615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109089,8.077537,10.276252,6.114536,6.263756,8.592278,6.632142,5.617401,3.798185,7.627045,10.475673,...,4.746759,5.820802,8.031175,6.308907,11.535426,6.661593,6.838952,7.384663,5.610963,8.048367
103176,7.166143,9.615435,6.836252,5.507424,8.735336,6.767973,6.649041,4.822174,7.817770,10.861289,...,4.078755,5.786654,8.127499,6.688908,11.910107,7.323802,6.889108,7.403089,6.744659,7.426063
102357,7.540244,9.095864,6.917348,6.168650,8.682271,6.479260,6.619081,4.656674,7.842086,10.577948,...,3.872740,5.898842,8.187308,6.619353,11.328832,6.578694,6.974518,7.468730,5.825379,7.680205
107362,6.883413,8.918684,6.115443,5.905714,8.505625,5.880857,6.616853,4.357334,7.870044,10.711813,...,4.641332,6.335894,8.406277,6.488221,11.888595,6.983108,6.811664,7.398692,5.688082,7.774392


In [4]:
import pickle

with open('list_groups_LC.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT',

In [5]:
list_groups = pd.Series(list_groups)

In [6]:
df_all_data = pd.read_pickle('df_all_data_Lisbon_Coimbra_threshold.pkl')
df_Coimbra = df_all_data[df_all_data['Lab_name'] == 'Coimbra']
y_Coimbra = df_Coimbra['Groups']
df_Lisbon = df_all_data[df_all_data['Lab_name'] == 'Lisbon']
y_Lisbon = df_Lisbon['Groups']
cols_to_drop = ['Lab', 'Sample code', 'Groups', 'Age', 'Gender', 'Lab_name']
df_coimbra_dc = df_Coimbra.drop(columns=cols_to_drop)
df_Lisbon_dc = df_Lisbon.drop(columns=cols_to_drop)
print("Same features:", all(df_coimbra_dc.columns == df_Lisbon_dc.columns))

Same features: True


In [7]:
df_coimbra_dc

,A0A0A0MS15,A0A0B4J1X8,A0A0C4DH38,O00391,O00533,O14498,O14594,O14773,O15394,O43505,...,Q9NRN5,Q9NT99,Q9P121,Q9P2S2,Q9UBP4,Q9UBX5,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7
74,5.338670,9.311739,5.368342,6.363451,8.811500,6.429840,6.232016,4.855457,7.661486,10.622482,...,4.850699,5.800033,8.415112,6.759808,11.329084,6.049101,6.976387,7.458193,6.139852,6.967906
75,6.571228,8.986320,5.400070,6.237011,8.556280,6.324377,6.199996,5.496839,7.435754,10.552957,...,4.335769,6.067658,8.203358,6.722548,11.352380,6.357804,6.761179,7.385923,6.398954,7.078834
76,6.168520,9.322539,6.493969,6.427135,9.248172,6.311663,6.820753,4.914124,7.922674,10.808787,...,4.894512,6.021373,8.712379,6.741656,11.591747,6.424029,7.325251,7.835930,6.582637,7.505629
77,7.464734,9.812152,7.210827,6.346462,8.885913,6.343495,6.712692,4.947535,7.917599,10.542935,...,5.383113,5.548655,8.368140,6.500428,11.666797,6.222213,6.355367,7.287186,6.226406,6.764420
78,5.200661,9.718603,6.962723,6.468935,9.069160,6.454995,6.894854,4.204978,7.255576,10.664821,...,5.042412,6.015781,8.467292,6.067114,11.768011,6.594716,7.064355,7.343239,6.230420,7.659339
79,6.080417,9.116656,6.068385,5.989673,8.741073,6.376876,6.712912,4.492783,7.202143,10.756181,...,4.639875,5.923705,8.565658,6.868131,11.799646,6.323964,7.183040,7.686171,6.173755,7.017710
80,6.452478,9.069530,6.511220,6.452959,8.528989,6.739443,6.402576,4.868494,7.410222,10.681379,...,4.933927,6.038425,8.317399,6.656554,11.519843,6.304646,6.712032,7.335998,6.424015,7.993810
81,6.248379,9.583534,6.165206,6.264426,8.599954,6.677551,6.555058,5.721783,7.761059,10.735488,...,5.188793,6.080809,8.294180,6.557016,11.373409,5.980238,6.957938,7.421526,6.570718,8.275324
82,6.457701,9.630560,6.603267,6.517710,9.056933,6.868230,6.649946,4.766001,7.737261,10.769706,...,4.754866,6.008901,8.469768,6.735170,11.248604,6.273708,6.913189,7.510329,5.884200,6.478033
83,4.231041,8.628099,4.587497,5.716072,9.127873,6.015469,7.389291,4.926962,7.582466,10.892209,...,4.915449,6.230405,8.384715,6.565285,11.583666,6.443028,7.160134,7.625658,6.226703,6.323552


Training: Coimbra Dataset | Testing: Lisbon Dataset
This section describes the application of Partial Least Squares Discriminant Analysis (PLS-DA) to evaluate the generalizability of the protein signature across different clinical centers (multicentric validation).

In [8]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

# =================================================================
# DATA PREPARATION (Coimbra as Training | Lisbon as External Test)
# =================================================================
X_train = df_coimbra_dc.copy()
y_train = np.array(y_Coimbra)

X_test = df_Lisbon_dc.copy()
y_test = np.array(y_Lisbon)

y_train_bin = (y_train == "MCI-AD").astype(int)
y_test_bin = (y_test == "MCI-AD").astype(int)

# ======================
# GLOBAL PARAMETERS
# ======================
thresholds = [0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# STORAGE
all_results = []
feature_counter = {}   # To track frequency of EVERY selected protein
threshold_counter = {} # To track frequency of best thresholds

def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# SEED LOOP
# ======================
for seed in seeds:
    # Internal CV setup
    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)
    threshold_scores = []

    for t in thresholds:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_subtrain, y_subtrain = X_train.iloc[train_idx], y_train[train_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

            # Bootstrap Feature Importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):
                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)
                rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)
            ranking_df = pd.DataFrame({"protein": feature_names, "importance": mean_importance}).sort_values("importance", ascending=False)
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            selected_features = ranking_df[ranking_df["cum_importance"] <= t]["protein"]
            if len(selected_features) == 0: selected_features = ranking_df["protein"].iloc[:1]

            rf_fold = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed)
            rf_fold.fit(X_subtrain[selected_features], y_subtrain)
            y_pred = rf_fold.predict(X_val[selected_features])
            fold_scores.append(matthews_corrcoef(y_val, y_pred))

        threshold_scores.append(np.mean(fold_scores))

    # Best threshold for this seed
    best_threshold = thresholds[np.argmax(threshold_scores)]
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1
    
    # Final Training for this seed
    importance_matrix_final = np.zeros((n_iterations, len(X_train.columns)))
    for i in range(n_iterations):
        X_boot, y_boot = stratified_bootstrap(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=i)
        rf.fit(X_boot, y_boot)
        importance_matrix_final[i] = rf.feature_importances_

    mean_imp_final = importance_matrix_final.mean(axis=0)
    ranking_final = pd.DataFrame({"protein": X_train.columns, "importance": mean_imp_final}).sort_values("importance", ascending=False)
    ranking_final["importance"] /= ranking_final["importance"].sum()
    ranking_final["cum_importance"] = ranking_final["importance"].cumsum()

    top_features = ranking_final[ranking_final["cum_importance"] <= best_threshold]["protein"]
    
    # TRACK EVERY FEATURE SELECTED IN THIS SEED
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # External Test (Lisbon)
    rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
    rf_final.fit(X_train[top_features], y_train)
    y_pred_lisbon = rf_final.predict(X_test[top_features])
    ad_idx_label = np.where(rf_final.classes_ == "MCI-AD")[0][0]
    y_prob_lisbon = rf_final.predict_proba(X_test[top_features])[:, ad_idx_label]

    seed_mcc = matthews_corrcoef(y_test, y_pred_lisbon)
    seed_auc = roc_auc_score(y_test_bin, y_prob_lisbon)

    # PER-SEED REPORT
    print(f"\n>> SEED {seed} RESULTS:")
    print(f"   Best Threshold: {best_threshold}")
    print(f"   Features Selected: {len(top_features)}")
    print(f"   Test MCC: {seed_mcc:.3f}")
    print(f"   Test AUC: {seed_auc:.3f}")

    all_results.append({
        "model": "RF","seed": seed, "mcc": seed_mcc, "auc": seed_auc, 
        "n_features": len(top_features), "threshold": best_threshold
    })

# ======================
# FINAL AGGREGATED REPORT
# ======================
df_res = pd.DataFrame(all_results)

print("\n" + "="*30)
print("FINAL SUMMARY (OVER ALL SEEDS)")
print("="*30)
print(f"Global Mean MCC: {df_res['mcc'].mean():.3f} (±{df_res['mcc'].std():.3f})")
print(f"Global Mean AUC: {df_res['auc'].mean():.3f} (±{df_res['auc'].std():.3f})")
print(f"Avg Features Selected: {df_res['n_features'].mean():.1f}")

print("\n--- Feature Frequency (Top 20) ---")
# Normalized frequency (0.0 to 1.0)
feat_freq_CoimbraonLisbon = (pd.Series(feature_counter).sort_values(ascending=False) / len(seeds))
print(feat_freq_CoimbraonLisbon.head(20))

print("\n--- Threshold Frequency ---")
thresh_freq = (pd.Series(threshold_counter).sort_index() / len(seeds))
print(thresh_freq)


>> SEED 0 RESULTS:
   Best Threshold: 0.7
   Features Selected: 33
   Test MCC: 0.380
   Test AUC: 0.724

>> SEED 1 RESULTS:
   Best Threshold: 0.8
   Features Selected: 49
   Test MCC: 0.380
   Test AUC: 0.776

>> SEED 2 RESULTS:
   Best Threshold: 0.7
   Features Selected: 34
   Test MCC: 0.380
   Test AUC: 0.727

>> SEED 3 RESULTS:
   Best Threshold: 0.8
   Features Selected: 49
   Test MCC: 0.364
   Test AUC: 0.718

>> SEED 4 RESULTS:
   Best Threshold: 0.8
   Features Selected: 51
   Test MCC: 0.380
   Test AUC: 0.693

>> SEED 5 RESULTS:
   Best Threshold: 0.7
   Features Selected: 33
   Test MCC: 0.397
   Test AUC: 0.738

>> SEED 6 RESULTS:
   Best Threshold: 0.8
   Features Selected: 50
   Test MCC: 0.380
   Test AUC: 0.749

>> SEED 7 RESULTS:
   Best Threshold: 0.8
   Features Selected: 49
   Test MCC: 0.380
   Test AUC: 0.692

>> SEED 8 RESULTS:
   Best Threshold: 0.9
   Features Selected: 83
   Test MCC: 0.364
   Test AUC: 0.798

>> SEED 9 RESULTS:
   Best Threshold: 0.8
   

Dataset: Coimbra + Lisbon (Merged)
In this section, a Partial Least Squares Discriminant Analysis (Random Forest) is performed on the combined clinical proteomic dataset.

In [9]:
print(len(feat_freq_CoimbraonLisbon))
feat_freq_CoimbraonLisbon.to_pickle('feat_freq_CoimbraonLisbon_rf_threshold.pkl')
df_res.to_pickle('results_CoimbraonLisbon_rf_threshold.pkl')

83


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ======================
# DATI
# ======================
X = data_log2.copy()
y = np.array(list_groups)

# binarizza per AUC (importante!)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# PARAMETRI GLOBALI
# ======================
thresholds = [ 0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# ======================
# STORAGE
# ======================
all_results = []
feature_counter = {}
threshold_counter = {}

# ======================
# BOOTSTRAP
# ======================
def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y[indices]


# ======================
# LOOP SEED
# ======================
for seed in seeds:

    print(f"\n===== SEED {seed} =====")

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    y_train_bin = (y_train == "MCI-AD").astype(int)
    y_test_bin = (y_test == "MCI-AD").astype(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    threshold_scores = []

    # ======================
    # CV per scegliere threshold
    # ======================
    for t in thresholds:

        fold_scores = []

        for train_idx, val_idx in cv.split(X_train, y_train):

            X_subtrain = X_train.iloc[train_idx]
            y_subtrain = y_train[train_idx]

            X_val = X_train.iloc[val_idx]
            y_val = y_train[val_idx]

            y_val_bin = (y_val == "MCI-AD").astype(int)

            # ===== bootstrap importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):

                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)

                rf = RandomForestClassifier(
                    n_estimators=200,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=seed
                )

                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)

            ranking_df = pd.DataFrame({
                "protein": feature_names,
                "importance": mean_importance
            }).sort_values("importance", ascending=False)

            # normalizzazione
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            # selezione feature
            selected_features = ranking_df[
                ranking_df["cum_importance"] <= t
            ]["protein"]

            if len(selected_features) == 0:
                selected_features = ranking_df["protein"].iloc[:1]

            # train modello
            rf = RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=seed
            )

            rf.fit(X_subtrain[selected_features], y_subtrain)

            # valutazione
            y_pred = rf.predict(X_val[selected_features])
            mcc = matthews_corrcoef(y_val, y_pred)

            fold_scores.append(mcc)

        threshold_scores.append(np.mean(fold_scores))

    # ======================
    # best threshold
    # ======================
    best_threshold = thresholds[np.argmax(threshold_scores)]
    print("Best threshold:", best_threshold)

    # ======================
    # TRAIN FINALE (su tutto train)
    # ======================
    feature_names = X_train.columns
    importance_matrix = np.zeros((n_iterations, len(feature_names)))

    for i in range(n_iterations):

        X_boot, y_boot = stratified_bootstrap(X_train, y_train)

        rf = RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            n_jobs=-1,
            random_state=i
        )

        rf.fit(X_boot, y_boot)
        importance_matrix[i] = rf.feature_importances_

    mean_importance = importance_matrix.mean(axis=0)

    ranking_df = pd.DataFrame({
        "protein": feature_names,
        "importance": mean_importance
    }).sort_values("importance", ascending=False)

    ranking_df["importance"] /= ranking_df["importance"].sum()
    ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

    top_features = ranking_df[
        ranking_df["cum_importance"] <= best_threshold
    ]["protein"]

    if len(top_features) == 0:
        top_features = ranking_df["protein"].iloc[:1]

    print("Numero feature:", len(top_features))

    # ======================
    # TEST
    # ======================
    rf_final = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42
    )

    rf_final.fit(X_train[top_features], y_train)

    y_pred = rf_final.predict(X_test[top_features])
    class_order = rf_final.classes_
    ad_index = np.where(class_order == "MCI-AD")[0][0]
    
    y_prob = rf_final.predict_proba(X_test[top_features])[:, ad_index]

    test_mcc = matthews_corrcoef(y_test, y_pred)
    test_auc = roc_auc_score(y_test_bin, y_prob)
    print("Class order:", rf_final.classes_)
    print("Test MCC:", test_mcc)
    print("Test AUC:", test_auc)

    # ======================
    # SALVATAGGIO
    # ======================
    all_results.append({
        "seed": seed,
        "mcc": test_mcc,
        "auc": test_auc,
        "n_features": len(top_features),
        "best_threshold": best_threshold
    })

    # frequenza feature
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # frequenza threshold
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1


# ======================
# RISULTATI FINALI
# ======================
df_results = pd.DataFrame(all_results)

print("\n===== FINAL RESULTS =====")
print("Mean MCC:", df_results["mcc"].mean())
print("Mean AUC:", df_results["auc"].mean())

# frequenze
feature_freq = pd.Series(feature_counter).sort_values(ascending=False) / len(seeds)
threshold_freq = pd.Series(threshold_counter).sort_index() / len(seeds)

print("\nTop features (frequency):")
print(feature_freq.head(20))

print("\nThreshold frequency:")
print(threshold_freq)


===== SEED 0 =====
Best threshold: 0.8
Numero feature: 81
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9225998461903602
Test AUC: 1.0

===== SEED 1 =====
Best threshold: 0.9
Numero feature: 122
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9230769230769231
Test AUC: 0.9807692307692308

===== SEED 2 =====
Best threshold: 0.8
Numero feature: 75
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.761306166786451
Test AUC: 0.9871794871794872

===== SEED 3 =====
Best threshold: 0.9
Numero feature: 127
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9230769230769231
Test AUC: 0.9615384615384616

===== SEED 4 =====
Best threshold: 0.8
Numero feature: 78
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.9225998461903602
Test AUC: 1.0

===== SEED 5 =====
Best threshold: 0.8
Numero feature: 84
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8498365855987975
Test AUC: 1.0

===== SEED 6 =====
Best threshold: 0.8
Numero feature: 78
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8498365855987975
Test AUC: 1.0

===== SE

In [11]:
print(len(feature_freq))
feature_freq.to_pickle('feature_freq_LC_rf_threshold.pkl')
df_results = pd.DataFrame(all_results)
df_results.to_pickle('results_LC_rf_threshold.pkl')

157


In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ======================
# DATI
# ======================
X = data_log2.copy()
y = np.array(list_groups)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# PARAMETRI
# ======================
thresholds = [0.7, 0.8, 0.9]
n_iterations = 20
n_splits = 5

# ======================
# BOOTSTRAP
# ======================
def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y[indices]

# ======================
# CV
# ======================
cv = StratifiedShuffleSplit(n_splits=n_splits, test_size=0.3, random_state=42)

threshold_scores = []

# ======================
# SCELTA THRESHOLD (CV)
# ======================
for t in thresholds:

    fold_mcc = []
    fold_auc = []

    for train_idx, val_idx in cv.split(X, y):

        X_sub = X.iloc[train_idx]
        y_sub = y[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y[val_idx]
        y_val_bin = (y_val == "MCI-AD").astype(int)

        # ===== bootstrap importance
        feature_names = X_sub.columns
        importance_matrix = np.zeros((n_iterations, len(feature_names)))

        for i in range(n_iterations):
            X_boot, y_boot = stratified_bootstrap(X_sub, y_sub)

            rf = RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                n_jobs=-1,
                random_state=i
            )

            rf.fit(X_boot, y_boot)
            importance_matrix[i] = rf.feature_importances_

        mean_importance = importance_matrix.mean(axis=0)

        ranking_df = pd.DataFrame({
            "protein": feature_names,
            "importance": mean_importance
        }).sort_values("importance", ascending=False)

        ranking_df["importance"] /= ranking_df["importance"].sum()
        ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

        selected_features = ranking_df[
            ranking_df["cum_importance"] <= t
        ]["protein"]

        if len(selected_features) == 0:
            selected_features = ranking_df["protein"].iloc[:1]

        # modello
        rf_model = RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42
        )

        rf_model.fit(X_sub[selected_features], y_sub)

        y_pred = rf_model.predict(X_val[selected_features])

        # ✅ FIX AUC
        class_order = rf_model.classes_
        ad_index = np.where(class_order == "MCI-AD")[0][0]
        y_prob = rf_model.predict_proba(X_val[selected_features])[:, ad_index]

        fold_mcc.append(matthews_corrcoef(y_val, y_pred))
        fold_auc.append(roc_auc_score(y_val_bin, y_prob))

    threshold_scores.append(np.mean(fold_mcc))

# ======================
# BEST THRESHOLD
# ======================
best_threshold = thresholds[np.argmax(threshold_scores)]
print("Best threshold:", best_threshold)

# ======================
# PERFORMANCE FINALE (CV)
# ======================
all_mcc = []
all_auc = []

# ✅ FREQUENZA FEATURE
feature_counts = pd.Series(0, index=X.columns)

for train_idx, val_idx in cv.split(X, y):

    X_sub = X.iloc[train_idx]
    y_sub = y[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y[val_idx]
    y_val_bin = (y_val == "MCI-AD").astype(int)

    # bootstrap importance
    importance_matrix = np.zeros((n_iterations, X.shape[1]))

    for i in range(n_iterations):
        X_boot, y_boot = stratified_bootstrap(X_sub, y_sub)

        rf = RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            n_jobs=-1,
            random_state=i
        )

        rf.fit(X_boot, y_boot)
        importance_matrix[i] = rf.feature_importances_

    mean_importance = importance_matrix.mean(axis=0)

    ranking_df = pd.DataFrame({
        "protein": X.columns,
        "importance": mean_importance
    }).sort_values("importance", ascending=False)

    ranking_df["importance"] /= ranking_df["importance"].sum()
    ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

    selected_features = ranking_df[
        ranking_df["cum_importance"] <= best_threshold
    ]["protein"]

    if len(selected_features) == 0:
        selected_features = ranking_df["protein"].iloc[:1]

    # ✅ conteggio frequenza
    feature_counts[selected_features] += 1

    rf_model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    )

    rf_model.fit(X_sub[selected_features], y_sub)

    y_pred = rf_model.predict(X_val[selected_features])

    # ✅ FIX AUC
    class_order = rf_model.classes_
    ad_index = np.where(class_order == "MCI-AD")[0][0]
    y_prob = rf_model.predict_proba(X_val[selected_features])[:, ad_index]

    all_mcc.append(matthews_corrcoef(y_val, y_pred))
    all_auc.append(roc_auc_score(y_val_bin, y_prob))

# ======================
# METRICHE FINALI
# ======================
print("\n=== VALIDATION PERFORMANCE ===")
print(f"MCC: {np.mean(all_mcc):.4f} ± {np.std(all_mcc):.4f}")
print(f"AUC: {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}")

# ======================
# FREQUENZA FEATURE
# ======================
feature_freq = (feature_counts / n_splits).sort_values(ascending=False)

print("\n=== FEATURE SELECTION FREQUENCY ===")
print(feature_freq)

# ======================
# FEATURE IMPORTANCE FINALE
# ======================
importance_matrix = np.zeros((n_iterations, X.shape[1]))

for i in range(n_iterations):
    X_boot, y_boot = stratified_bootstrap(X, y)

    rf = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        n_jobs=-1,
        random_state=i
    )

    rf.fit(X_boot, y_boot)
    importance_matrix[i] = rf.feature_importances_

mean_importance = importance_matrix.mean(axis=0)

ranking_df = pd.DataFrame({
    "protein": X.columns,
    "importance": mean_importance
}).sort_values("importance", ascending=False)

ranking_df["importance"] /= ranking_df["importance"].sum()
ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

print("\n=== GLOBAL RF RANKING ===")
print(ranking_df)

# salva
ranking_df.to_csv("rf_global_ranking.csv", index=False)
feature_freq.to_csv("rf_feature_frequency.csv")

Best threshold: 0.8

=== VALIDATION PERFORMANCE ===
MCC: 0.9062 ± 0.0213
AUC: 0.9789 ± 0.0113

=== FEATURE SELECTION FREQUENCY ===
Protein.Group
P40925    1.0
P14618    1.0
P23142    1.0
P02647    1.0
Q86UX2    1.0
         ... 
P05452    0.0
P05154    0.0
P04264    0.0
P04217    0.0
P10451    0.0
Length: 201, dtype: float64

=== GLOBAL RF RANKING ===
    protein  importance  cum_importance
110  P14618    0.077021        0.077021
55   P02766    0.042323        0.119344
127  P25311    0.041901        0.161245
158  Q13449    0.038572        0.199817
168  Q16270    0.038223        0.238040
..      ...         ...             ...
18   P00748    0.000766        0.997206
26   P01031    0.000743        0.997949
60   P04004    0.000721        0.998671
95   P08697    0.000694        0.999364
56   P02768    0.000636        1.000000

[201 rows x 3 columns]


In [13]:
feature_freq.to_pickle('feature_2_rf_LC_threshold.pkl')

Training: Lisbon Dataset | Testing: Coimbra Dataset
This section describes the "Inverse Validation" approach using Partial Least Squares Discriminant Analysis (PLS-DA). By swapping the roles of the two cohorts, we test the consistency and symmetry of our protein biomarkers.

In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

# =================================================================
# DATA PREPARATION (Lisbon as Training | Coimbra as External Test)
# =================================================================
print('ok')
X_train = df_Lisbon_dc.copy()
y_train = np.array(y_Lisbon)

X_test = df_coimbra_dc.copy()
y_test = np.array(y_Coimbra)

y_train_bin = (y_train == "MCI-AD").astype(int)
y_test_bin = (y_test == "MCI-AD").astype(int)

# ======================
# GLOBAL PARAMETERS
# ======================
thresholds = [0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# STORAGE
all_results = []
feature_counter = {}   # To track frequency of EVERY selected protein
threshold_counter = {} # To track frequency of best thresholds

def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]
    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)
    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)
    return X.iloc[indices], y[indices]

# ======================
# SEED LOOP
# ======================
for seed in seeds:
    # Internal CV setup
    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)
    threshold_scores = []

    for t in thresholds:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_subtrain, y_subtrain = X_train.iloc[train_idx], y_train[train_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

            # Bootstrap Feature Importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):
                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)
                rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)
            ranking_df = pd.DataFrame({"protein": feature_names, "importance": mean_importance}).sort_values("importance", ascending=False)
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            selected_features = ranking_df[ranking_df["cum_importance"] <= t]["protein"]
            if len(selected_features) == 0: selected_features = ranking_df["protein"].iloc[:1]

            rf_fold = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed)
            rf_fold.fit(X_subtrain[selected_features], y_subtrain)
            y_pred = rf_fold.predict(X_val[selected_features])
            fold_scores.append(matthews_corrcoef(y_val, y_pred))

        threshold_scores.append(np.mean(fold_scores))

    # Best threshold for this seed
    best_threshold = thresholds[np.argmax(threshold_scores)]
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1
    
    # Final Training for this seed
    importance_matrix_final = np.zeros((n_iterations, len(X_train.columns)))
    for i in range(n_iterations):
        X_boot, y_boot = stratified_bootstrap(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=i)
        rf.fit(X_boot, y_boot)
        importance_matrix_final[i] = rf.feature_importances_

    mean_imp_final = importance_matrix_final.mean(axis=0)
    ranking_final = pd.DataFrame({"protein": X_train.columns, "importance": mean_imp_final}).sort_values("importance", ascending=False)
    ranking_final["importance"] /= ranking_final["importance"].sum()
    ranking_final["cum_importance"] = ranking_final["importance"].cumsum()

    top_features = ranking_final[ranking_final["cum_importance"] <= best_threshold]["protein"]
    
    # TRACK EVERY FEATURE SELECTED IN THIS SEED
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # External Test (Lisbon)
    rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
    rf_final.fit(X_train[top_features], y_train)
    y_pred_lisbon = rf_final.predict(X_test[top_features])
    ad_idx_label = np.where(rf_final.classes_ == "MCI-AD")[0][0]
    y_prob_lisbon = rf_final.predict_proba(X_test[top_features])[:, ad_idx_label]

    seed_mcc = matthews_corrcoef(y_test, y_pred_lisbon)
    seed_auc = roc_auc_score(y_test_bin, y_prob_lisbon)

    # PER-SEED REPORT
    print(f"\n>> SEED {seed} RESULTS:")
    print(f"   Best Threshold: {best_threshold}")
    print(f"   Features Selected: {len(top_features)}")
    print(f"   Test MCC: {seed_mcc:.3f}")
    print(f"   Test AUC: {seed_auc:.3f}")

    all_results.append({
        "seed": seed, "mcc": seed_mcc, "auc": seed_auc, 
        "n_features": len(top_features), "threshold": best_threshold
    })

# ======================
# FINAL AGGREGATED REPORT
# ======================
df_res = pd.DataFrame(all_results)

print("\n" + "="*30)
print("FINAL SUMMARY (OVER ALL SEEDS)")
print("="*30)
print(f"Global Mean MCC: {df_res['mcc'].mean():.3f} (±{df_res['mcc'].std():.3f})")
print(f"Global Mean AUC: {df_res['auc'].mean():.3f} (±{df_res['auc'].std():.3f})")
print(f"Avg Features Selected: {df_res['n_features'].mean():.1f}")

print("\n--- Feature Frequency (Top 20) ---")
# Normalized frequency (0.0 to 1.0)
feat_freq_LisbononCoimbra = (pd.Series(feature_counter).sort_values(ascending=False) / len(seeds))
print(feat_freq_LisbononCoimbra.head(20))

print("\n--- Threshold Frequency ---")
thresh_freq = (pd.Series(threshold_counter).sort_index() / len(seeds))
print(thresh_freq)

ok

>> SEED 0 RESULTS:
   Best Threshold: 0.7
   Features Selected: 49
   Test MCC: 0.752
   Test AUC: 0.985

>> SEED 1 RESULTS:
   Best Threshold: 0.7
   Features Selected: 51
   Test MCC: 0.726
   Test AUC: 0.984

>> SEED 2 RESULTS:
   Best Threshold: 0.7
   Features Selected: 52
   Test MCC: 0.807
   Test AUC: 0.972

>> SEED 3 RESULTS:
   Best Threshold: 0.7
   Features Selected: 49
   Test MCC: 0.866
   Test AUC: 0.988

>> SEED 4 RESULTS:
   Best Threshold: 0.8
   Features Selected: 79
   Test MCC: 0.701
   Test AUC: 1.000

>> SEED 5 RESULTS:
   Best Threshold: 0.7
   Features Selected: 51
   Test MCC: 0.836
   Test AUC: 0.986

>> SEED 6 RESULTS:
   Best Threshold: 0.7
   Features Selected: 51
   Test MCC: 0.836
   Test AUC: 0.982

>> SEED 7 RESULTS:
   Best Threshold: 0.7
   Features Selected: 52
   Test MCC: 0.701
   Test AUC: 0.983

>> SEED 8 RESULTS:
   Best Threshold: 0.8
   Features Selected: 82
   Test MCC: 0.701
   Test AUC: 0.991

>> SEED 9 RESULTS:
   Best Threshold: 0.8


In [15]:
print(len(feat_freq_LisbononCoimbra))
feat_freq_LisbononCoimbra.to_pickle('feat_freq_LisbononCoimbra_rf_threshold.pkl')
df_res.to_pickle('results_LisbononCoimbra_rf_threshold.pkl')

126
